# Attention language model

Start from the same Tiny Shakespeare token stream as the bigram model, then inspect token embeddings before adding attention.

In [8]:
import random
import sys
from pathlib import Path

import torch
from torch import nn

repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.dataset import get_batch, load_tiny_shakespeare_tokens, split_token_stream

## Prepare token streams and batches

In [9]:
tokens, vocab, _ = load_tiny_shakespeare_tokens(repo_root / "data")
train_tokens, validation_tokens = split_token_stream(tokens)

vocab_size = len(vocab)
block_size = 8
batch_size = 32
n_embd = 32
head_size = 16

random.seed(42)
x_batch, y_batch = get_batch("train", train_tokens, validation_tokens, block_size, batch_size)
x_batch = torch.tensor(x_batch, dtype=torch.long)
y_batch = torch.tensor(y_batch, dtype=torch.long)

## Model scaffold

In [10]:
class AttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd):
        super().__init__()
        self.head = self.Head(n_embd, head_size)  # (B, T, H)
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.positional_embedding_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(head_size, vocab_size)
        # self.positional_embedding_table = nn.Parameter(torch.randn(vocab_size, head_size))

    def forward(self, idx):
        B, T = idx.shape
        token_embeddings = self.token_embedding_table(idx)  # (B, T, C)
        positions = torch.arange(T).unsqueeze(0).repeat(B, 1).to(device)  # (B, T)
        positional_embeddings = self.positional_embedding_table(positions)  # (B, T, C)
        x = token_embeddings + positional_embeddings  # (B, T, C)
        x = self.head(x)  # (B, T, H)
        return self.lm_head(x)  # (B, T, V)

    # # x: [B, T, C]
    # q,k,v: [B, T, H]
    # out: [B, T, H]
    class Head(nn.Module):
        def __init__(self, n_embd, head_size):
            super().__init__()
            self.head_size = head_size
            self.query = nn.Linear(n_embd, head_size)
            self.key = nn.Linear(n_embd, head_size)
            self.value = nn.Linear(n_embd, head_size)

        def forward(self, x):
            q = self.query(x)  # (B, T, H)
            k = self.key(x)  # (B, T, H)
            v = self.value(x)  # (B, T, H)

            output = self.attention(q, k, v)  # (B, T, H)
            return output

        def attention(self, q, k, v):
            # Compute attention scores
            attn_scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_size**0.5)  # (B, T, T)
            # Apply masking to attention scores
            T = attn_scores.shape[-1]
            mask = torch.triu(torch.ones(T, T), diagonal=1).bool()  # (T, T)
            attn_scores = attn_scores.masked_fill(mask, -float("inf"))
            # Normalize attention scores
            attn_weights = torch.softmax(attn_scores, dim=-1)  # (B, T, T)
            # Compute attention outputs
            out = torch.matmul(attn_weights, v)  # (B, T, H)
            return out


model = AttentionLanguageModel(vocab_size, n_embd)
head = model.Head(n_embd, head_size)
token_embeddings = model.token_embedding_table(x_batch)
print(f"Token embedding shape (B, T, C): {tuple(token_embeddings.shape)}")
attention = head(token_embeddings)
print(f"Attention shape (B, T, H): {tuple(attention.shape)}")

Token embedding shape (B, T, C): (32, 8, 32)
Attention shape (B, T, H): (32, 8, 16)


# TODO:

```
single causal head
→ multi-head attention
→ positional embeddings integrated into the model
→ train on Tiny Shakespeare
→ record validation loss
→ generate samples
→ compare against bigram
```

In [11]:
# multi-head attention
class multihead_attention(nn.Module):
    def __init__(self, n_head, d_model, d_k, d_v, dropout=0.1):
        super().__init__()
        self.n_head = n_head
        self.d_k = d_k
        self.d_v = d_v

        self.w_qs = nn.Linear(d_model, n_head * d_k)
        self.w_ks = nn.Linear(d_model, n_head * d_k)
        self.w_vs = nn.Linear(d_model, n_head * d_v)
        nn.init.normal_(self.w_qs.weight, mean=0, std=np.sqrt(2.0 / (d_model + d_k)))
        nn.init.normal_(self.w_ks.weight, mean=0, std=np.sqrt(2.0 / (d_model + d_k)))
        nn.init.normal_(self.w_vs.weight, mean=0, std=np.sqrt(2.0 / (d_model + d_v)))

        self.attention = ScaledDotProductAttention(temperature=np.power(d_k, 0.5))
        self.layer_norm = nn.LayerNorm(d_model)

        self.fc = nn.Linear(n_head * d_v, d_model)
        nn.init.xavier_normal_(self.fc.weight)

        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        d_k, d_v, n_head = self.d_k, self.d_v, self.n_head
        sz_b, len_q, _ = q.size()
        sz_b, len_k, _ = k.size()
        sz_b, len_v, _ = v.size()

        residual = q

        q = self.w_qs(q).view(sz_b, len_q, n_head, d_k)
        k = self.w_ks(k).view(sz_b, len_k, n_head, d_k)
        v = self.w_vs(v).view(sz_b, len_v, n_head, d_v)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # scaled dot-product attention
        # q @ k^T: (B, h, T, d_k) @ (B, h, d_k, T)
        #          -> (B, h, T, T)
        scores = q @ k.transpose(-2, -1)
        scores = scores / (d_k**0.5)

        # causal mask
        T = scores.size(-1)
        causal_mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool),
            diagonal=1,
        )
        scores = scores.masked_fill(causal_mask, float("-inf"))

        # normalize, then retrieve values
        weights = torch.softmax(scores, dim=-1)  # (B, h, T, T)
        output = weights @ v  # (B, h, T, d_v)

        # put heads beside each other again
        output = output.transpose(1, 2).contiguous()  # (B, T, h, d_v)
        output = output.view(sz_b, len_q, n_head * d_v)
        # (B, T, h*d_v)

        # mix head outputs back into model space
        output = self.fc(output)  # (B, T, d_model)
        output = self.dropout(output)

        # residual path + normalization
        output = self.layer_norm(output + residual)

        return output, weights